In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install dependencies\n
!pip -q install ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 120.2 MB/s eta 0:00:00


In [ ]:
# Roboflow download (set your API key)\n
from roboflow import Roboflow
import os

ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', 'P1wt7BrRmYauvnFdZDKK')  # or set manually\n
if not ROBOFLOW_API_KEY:
    raise ValueError('Set ROBOFLOW_API_KEY in environment or edit this cell.')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('gao-shou-zheng-b6xqc').project('solar-panel-0swal')
dataset = project.version(3).download('yolov8')
print('Dataset path:', dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to solar-panel-3 in yolov8:: 100%|██████████| 4572/4572 [00:00<00:00, 8263.81it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset path: /content/solar-panel-3


In [ ]:
# Train YOLOv8 detection model (STM32-friendly settings)
from ultralytics import YOLO

# YOLOv8n is the smallest; good for MCU export
model = YOLO('yolov8n.pt')

results = model.train(
    data=f'{dataset.location}/data.yaml',
    imgsz=320,
    epochs=80,
    batch=16,
    device=0,
    project='runs_solar',
    name='yolov8n_solar_320'
)

Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/solar-panel-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_solar_320, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspecti

In [ ]:
# Validate\n
metrics = model.val()
print(metrics)

Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1361.8±648.1 MB/s, size: 51.8 KB)
val: Scanning /content/solar-panel-3/valid/labels.cache... 310 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 310/310 673.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 10.3it/s 1.9s
                   all        310        455        0.8      0.687      0.735      0.419
                 cover         15         29      0.894      0.414      0.542      0.212
                 crack        177        188      0.901      0.875      0.934      0.559
                  dust         67        144      0.796      0.649      0.728      0.453
                normal         67         94       0.61      0.809      0.736       0.45
Speed: 0.3ms preprocess, 2.0

In [ ]:
# Export to TFLite INT8 (uses dataset for calibration)
model.export(format='tflite', int8=True, data=f'{dataset.location}/data.yaml', imgsz=320)

Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/runs_solar/yolov8n_solar_320/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 8, 2100) (5.9 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx>=1.12.0,<2.0.0', 'onnx2tf>=1.26.3', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 20 packages in 2.77s
Prepared 11 packages in 3.86s
Installed 11 packages in 252ms
 + ai-edge-litert==2.1.0
 + backports-strenum==1.3.1
 + colorama==0.4.6
 + coloredlogs==15.0.1
 + humanfriendly==10.0
 + onnx==1.20.0
 + onnx-graphsurgeon==0.5.8
 + onnx2tf==1.28.8
 + onnxruntime-gpu==1.23.2
 + onnxslim==0.1.82
 + sng4onnx==1.0.4



Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 


ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 1.3s, saved as '/content/runs_solar/yolov8n_solar_320/weights/best.onnx' (11.6 MB)
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /content/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 47.0files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/runs_solar/yolov8n_solar_320/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 320, 320, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 8, 2100), dtype=tf.float32, name=None)
Captures:
  133655008995344: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  133655008993808: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  133655008994576: TensorSpec(shape=(16,), dtype=tf.float32, name=None)
  133655008998224: TensorSpec(shape=(4, 2), dtyp

'/content/runs_solar/yolov8n_solar_320/weights/best_saved_model/best_int8.tflite'

In [ ]:
# Copy artifacts to Drive
!mkdir -p /content/drive/MyDrive/solar_dataset/roboflow_detection
!cp -r runs_solar/yolov8n_solar_320 /content/drive/MyDrive/solar_dataset/roboflow_detection/
!cp runs_solar/yolov8n_solar_320/weights/best.pt /content/drive/MyDrive/solar_dataset/roboflow_detection/
!cp {dataset.location}/data.yaml /content/drive/MyDrive/solar_dataset/roboflow_detection/
print('Saved to Drive: /content/drive/MyDrive/solar_dataset/roboflow_detection/')


Saved to Drive: /content/drive/MyDrive/solar_dataset/roboflow_detection/
